# APIs públicas · Peticiones HTTP

Consumir APIs REST reales con `requests`: parámetros, cabeceras, códigos de estado, errores y tiempos de espera, reintentos, los verbos de un CRUD, paginación, límites de uso, caché y peticiones en paralelo. Todas las APIs de esta sección son públicas y **no necesitan clave**: [httpbin](https://httpbin.org) (devuelve lo que le envías), [JSONPlaceholder](https://jsonplaceholder.typicode.com) (una API falsa para practicar), [PokéAPI](https://pokeapi.co) y la [API de GitHub](https://docs.github.com/rest) sin autenticar.

Necesita conexión a internet. Las salidas guardadas son las de la fecha en que se ejecutó el notebook: los datos en vivo pueden cambiar, así que las comprobaciones miran la forma de las respuestas y no los valores concretos. Conviene no abusar de estos servicios gratuitos: la API de GitHub, por ejemplo, admite 60 peticiones por hora sin clave.

## 1. Primera petición

Hacer un `GET` a `https://httpbin.org/get` con los parámetros `isla=La Palma` y `año=2026` y una cabecera propia. Mostrar el código de estado, el tipo de contenido, la URL final (con los parámetros codificados) y lo que httpbin dice haber recibido. Comprobar que los parámetros llegan intactos.

In [1]:
import requests

respuesta = requests.get(
    "https://httpbin.org/get",
    params={"isla": "La Palma", "año": 2026},
    headers={"X-Curso": "ejercicios-python"},
    timeout=15,
)
print(respuesta.status_code, respuesta.reason, "|", respuesta.headers["Content-Type"])
print(respuesta.url)
datos = respuesta.json()
print("Parámetros recibidos:", datos["args"])
print("Cabecera propia:", datos["headers"]["X-Curso"])
assert respuesta.ok and datos["args"] == {"isla": "La Palma", "año": "2026"}

200 OK | application/json
https://httpbin.org/get?isla=La+Palma&a%C3%B1o=2026
Parámetros recibidos: {'año': '2026', 'isla': 'La Palma'}
Cabecera propia: ejercicios-python


## 2. Una sesión reutilizable

Crear una `requests.Session` con un `User-Agent` propio (identificar el cliente es una norma de cortesía que piden muchas APIs) y una función `obtener_json(url, **params)` que use siempre un tiempo de espera, lance una excepción si el código de estado es de error y devuelva el JSON. Comprobar con `https://httpbin.org/headers` que la cabecera se envía en cada petición.

In [2]:
sesion = requests.Session()
sesion.headers["User-Agent"] = "ejercicios-python/1.0"


def obtener_json(url, **params):
    respuesta = sesion.get(url, params=params, timeout=15)
    respuesta.raise_for_status()
    return respuesta.json()


cabeceras = obtener_json("https://httpbin.org/headers")["headers"]
print(cabeceras["User-Agent"])
assert cabeceras["User-Agent"] == "ejercicios-python/1.0"

ejercicios-python/1.0


## 3. Errores y tiempos de espera

Provocar y capturar los tres fallos típicos: un código de error HTTP (`/status/404` de httpbin, que `raise_for_status` convierte en `HTTPError`), una respuesta que tarda más que el tiempo de espera (`/delay/3` con `timeout=1`) y un servidor que no existe (el dominio `.invalid` está reservado para eso). Mostrar el tipo de cada excepción.

In [3]:
casos = {
    "error 404": lambda: obtener_json("https://httpbin.org/status/404"),
    "demasiado lento": lambda: sesion.get("https://httpbin.org/delay/3", timeout=1),
    "servidor inexistente": lambda: sesion.get("https://api.ejemplo.invalid/datos", timeout=5),
}
capturadas = {}
for nombre, peticion in casos.items():
    try:
        peticion()
    except requests.RequestException as e:
        capturadas[nombre] = type(e).__name__
        print(f"{nombre:<21} -> {type(e).__name__}")
assert capturadas == {"error 404": "HTTPError", "demasiado lento": "ReadTimeout", "servidor inexistente": "ConnectionError"}

error 404             -> HTTPError


demasiado lento       -> ReadTimeout
servidor inexistente  -> ConnectionError


## 4. Reintentos automáticos

Montar en una sesión un `HTTPAdapter` con una política `Retry` de `urllib3`: hasta 3 reintentos con espera exponencial cuando el servidor responde 500, 502, 503 o 504. Probarla contra `/status/503`, que falla siempre, capturar el `RetryError` final y contar en el historial de reintentos cuántas veces se intentó.

In [4]:
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

politica = Retry(total=3, backoff_factor=0.2, status_forcelist=[500, 502, 503, 504], allowed_methods=["GET"])
con_reintentos = requests.Session()
adaptador = HTTPAdapter(max_retries=politica)
con_reintentos.mount("https://", adaptador)

try:
    con_reintentos.get("https://httpbin.org/status/503", timeout=15)
except requests.exceptions.RetryError as e:
    historial = e.args[0].reason
    print("Se rindió tras los reintentos:", type(e).__name__)
    print("Motivo:", historial)
assert politica.total == 3

Se rindió tras los reintentos: RetryError
Motivo: too many 503 error responses


## 5. Consultar una API REST

Descargar de JSONPlaceholder las 200 tareas (`/todos`) y contar, para cada usuario, cuántas tiene terminadas y cuántas pendientes. Pedir después solo las del usuario 3 con el parámetro de consulta `userId` y comprobar que coinciden con filtrar la lista completa en local.

In [5]:
from collections import Counter

tareas = obtener_json("https://jsonplaceholder.typicode.com/todos")
terminadas = Counter(t["userId"] for t in tareas if t["completed"])
pendientes = Counter(t["userId"] for t in tareas if not t["completed"])
print(len(tareas), "tareas")
print(f"{'usuario':>7} {'hechas':>7} {'pendientes':>10}")
for usuario in sorted(terminadas | pendientes):
    print(f"{usuario:>7} {terminadas[usuario]:>7} {pendientes[usuario]:>10}")

del_3 = obtener_json("https://jsonplaceholder.typicode.com/todos", userId=3)
assert del_3 == [t for t in tareas if t["userId"] == 3]

200 tareas
usuario  hechas pendientes
      1      11          9
      2       8         12
      3       7         13
      4       6         14
      5      12          8
      6       6         14
      7       9         11
      8      11          9
      9       8         12
     10      12          8


## 6. Crear, modificar y borrar

Hacer con JSONPlaceholder las cuatro operaciones de escritura de un CRUD: `POST` para crear una publicación (enviando JSON), `PUT` para reemplazarla, `PATCH` para cambiar un solo campo y `DELETE` para borrarla. La API no guarda nada de verdad, pero responde como si lo hiciera: mostrar el código de estado y la respuesta de cada una.

In [6]:
base = "https://jsonplaceholder.typicode.com/posts"
nueva = {"title": "Ruta por el Roque Nublo", "body": "Salida desde la Goleta.", "userId": 1}

operaciones = [
    ("POST", base, nueva),
    ("PUT", f"{base}/1", {**nueva, "id": 1}),
    ("PATCH", f"{base}/1", {"title": "Ruta por el Roque Nublo y el Bentayga"}),
    ("DELETE", f"{base}/1", None),
]
codigos = []
for metodo, url, cuerpo in operaciones:
    r = sesion.request(metodo, url, json=cuerpo, timeout=15)
    codigos.append(r.status_code)
    print(f"{metodo:<6} {r.status_code}  {r.json()}")
assert codigos == [201, 200, 200, 200]

POST   201  {'title': 'Ruta por el Roque Nublo', 'body': 'Salida desde la Goleta.', 'userId': 1, 'id': 101}


PUT    200  {'title': 'Ruta por el Roque Nublo', 'body': 'Salida desde la Goleta.', 'userId': 1, 'id': 1}
PATCH  200  {'userId': 1, 'id': 1, 'title': 'Ruta por el Roque Nublo y el Bentayga', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


DELETE 200  {}


## 7. Paginación

Las APIs que devuelven listas largas las reparten en páginas. PokéAPI indica el total en `count` y la URL de la página siguiente en `next`. Recorrer tres páginas de 20 elementos siguiendo `next` y comprobar que el resultado es el mismo que pedir una sola página de 60 con los parámetros `limit` y `offset`.

In [7]:
url = "https://pokeapi.co/api/v2/pokemon?limit=20"
nombres = []
for _ in range(3):
    pagina = obtener_json(url)
    nombres += [p["name"] for p in pagina["results"]]
    url = pagina["next"]
print("Total en la API:", pagina["count"])
print("Siguiente página:", url)
print(nombres[:5], "...", nombres[-3:])

de_golpe = [p["name"] for p in obtener_json("https://pokeapi.co/api/v2/pokemon", limit=60, offset=0)["results"]]
assert nombres == de_golpe and len(nombres) == 60

Total en la API: 1351
Siguiente página: https://pokeapi.co/api/v2/pokemon?offset=60&limit=20
['bulbasaur', 'ivysaur', 'venusaur', 'charmander', 'charmeleon'] ... ['growlithe', 'arcanine', 'poliwag']


## 8. Cabecera `Link` y límite de peticiones

GitHub pagina con la cabecera `Link`, que `requests` convierte en el diccionario `respuesta.links`. Pedir los colaboradores de `python/cpython` de 5 en 5, mostrar las URL de la página siguiente y la última, y deducir cuántas páginas hay. Leer también las cabeceras `X-RateLimit-*`, que dicen cuántas peticiones quedan sin autenticar.

In [8]:
from urllib.parse import parse_qs, urlparse

respuesta = sesion.get("https://api.github.com/repos/python/cpython/contributors", params={"per_page": 5}, timeout=15)
respuesta.raise_for_status()
print([c["login"] for c in respuesta.json()])
for relacion, enlace in respuesta.links.items():
    print(f"{relacion:>5}: {enlace['url']}")
paginas = int(parse_qs(urlparse(respuesta.links["last"]["url"]).query)["page"][0])
print("Páginas de 5 colaboradores:", paginas)

limite = int(respuesta.headers["X-RateLimit-Limit"])
restantes = int(respuesta.headers["X-RateLimit-Remaining"])
print("Límite por hora sin clave:", limite)
assert 0 <= restantes < limite and paginas > 1

['gvanrossum', 'vstinner', 'benjaminp', 'serhiy-storchaka', 'birkenfeld']
 next: https://api.github.com/repositories/81598961/contributors?per_page=5&page=2
 last: https://api.github.com/repositories/81598961/contributors?per_page=5&page=72
Páginas de 5 colaboradores: 72
Límite por hora sin clave: 60


## 9. Caché de respuestas

Repetir la misma petición gasta tiempo y cuota. Escribir `obtener_con_cache`, que guarde en un diccionario el JSON de cada URL y sus parámetros y solo llame a la API la primera vez. Contar las peticiones reales al pedir varias veces los mismos Pokémon y comparar con `functools.lru_cache`.

In [9]:
from functools import lru_cache

cache = {}
peticiones_reales = 0


def obtener_con_cache(url, **params):
    global peticiones_reales
    clave = (url, tuple(sorted(params.items())))
    if clave not in cache:
        peticiones_reales += 1
        cache[clave] = obtener_json(url, **params)
    return cache[clave]


pedidos = ["pikachu", "eevee", "pikachu", "snorlax", "eevee", "pikachu"]
pesos = {n: obtener_con_cache(f"https://pokeapi.co/api/v2/pokemon/{n}")["weight"] / 10 for n in pedidos}
print(pesos, "| pedidos:", len(pedidos), "| peticiones reales:", peticiones_reales)
assert peticiones_reales == len(set(pedidos))


@lru_cache(maxsize=32)
def altura(nombre):
    return obtener_json(f"https://pokeapi.co/api/v2/pokemon/{nombre}")["height"] / 10


for n in pedidos:
    altura(n)
print(altura.cache_info())
assert altura.cache_info().misses == 3

{'pikachu': 6.0, 'eevee': 6.5, 'snorlax': 460.0} | pedidos: 6 | peticiones reales: 3


CacheInfo(hits=3, misses=3, maxsize=32, currsize=3)


## 10. Peticiones en paralelo

Descargar la ficha de los 12 primeros Pokémon de forma secuencial y en paralelo con un `ThreadPoolExecutor` de 6 hilos (las peticiones pasan casi todo el tiempo esperando a la red, así que los hilos ayudan). Comprobar que ambos resultados son iguales y mostrar una tabla con el nombre, los tipos, la altura y el peso.

In [10]:
from concurrent.futures import ThreadPoolExecutor


def ficha(numero):
    datos = obtener_json(f"https://pokeapi.co/api/v2/pokemon/{numero}")
    return datos["id"], datos["name"], "/".join(t["type"]["name"] for t in datos["types"]), datos["height"] / 10, datos["weight"] / 10


secuencial = [ficha(n) for n in range(1, 13)]
with ThreadPoolExecutor(max_workers=6) as pool:
    paralelo = list(pool.map(ficha, range(1, 13)))
assert secuencial == paralelo
print(f"{'nº':>3} {'nombre':<11} {'tipos':<14} {'altura (m)':>10} {'peso (kg)':>9}")
for numero, nombre, tipos, alto, peso in paralelo:
    print(f"{numero:>3} {nombre:<11} {tipos:<14} {alto:>10} {peso:>9}")

 nº nombre      tipos          altura (m) peso (kg)
  1 bulbasaur   grass/poison          0.7       6.9
  2 ivysaur     grass/poison          1.0      13.0
  3 venusaur    grass/poison          2.0     100.0
  4 charmander  fire                  0.6       8.5
  5 charmeleon  fire                  1.1      19.0
  6 charizard   fire/flying           1.7      90.5
  7 squirtle    water                 0.5       9.0
  8 wartortle   water                 1.0      22.5
  9 blastoise   water                 1.6      85.5
 10 caterpie    bug                   0.3       2.9
 11 metapod     bug                   0.7       9.9
 12 butterfree  bug/flying            1.1      32.0
